[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/255ribeiro/python_build123d_basics/blob/master/docs/tuto_colab_build/build123d_interactive_exploration.ipynb)

# Explorando `show()` e `@interactive()`
## build123d para Arquitetos e Engenheiros
### Versão Google Colab

---

Este notebook explora funcionalidades novas do **`cadquery_simpleviewer`**: o parâmetro `flat_shading`/`angular_tolerance` do `show()` e o decorador `@interactive()`, que transforma qualquer função que devolve um sólido em um modelo com **sliders interativos**. Os modelos são construídos com o **build123d em modo álgebra** (`Box(...) - Cylinder(...)`, `fillet(arestas, raio)`, etc.).

> 💡 Este notebook **não faz parte da suíte automatizada de testes** do curso — os sliders exigem execução interativa. Rode as células no Colab ou localmente e avalie o resultado visualmente.

---

## Instalação

In [ ]:
import sys

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    import subprocess
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q",
         "cadquery-simpleviewer[build123d,interactive]"],
        check=True,
    )
    # build123d pulls in a newer ipython than Colab's kernel bootstrap
    # tolerates. Put Colab's version back on disk — do NOT restart the
    # runtime, the current kernel already has the working ipython loaded.
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q",
         "ipython==7.34.0", "--no-deps"],
        check=True,
    )

else:
    print("Not running in Colab, skipping package installation.")

## Importações

In [ ]:
import ipywidgets as widgets
import build123d as b3d
from cadquery_simpleviewer import show, interactive

## 1. `show()` básico

Verificação simples de que o `show()` renderiza normalmente um objeto do build123d criado em modo álgebra (sem `with BuildPart()`).

In [ ]:
box = b3d.Box(5, 3, 2)
show(box)

## 2. Diferença booleana — sombreamento plano vs. suave

O artefato relatado (manchas escuras/moiré) aparecia nas faces curvas criadas por um corte booleano, com `flat_shading=True`. Em modo álgebra, o corte e o friso (fillet) são expressos diretamente com operadores: `Box(...) - Cylinder(...)` e `fillet(arestas, raio)`.

In [ ]:
def caixa_furada_com_friso():
    caixa = b3d.Box(20, 20, 20)
    furo = b3d.Cylinder(5, 30)
    cortada = caixa - furo
    arestas_verticais = cortada.edges().filter_by(b3d.Axis.Z)
    return b3d.fillet(arestas_verticais, radius=2)

print("flat_shading=True (padrão antigo) — espera-se aspecto facetado/moiré no furo/friso")
show(caixa_furada_com_friso(), flat_shading=True)

In [ ]:
print("flat_shading=False (novo padrão) — espera-se superfícies curvas suaves")
show(caixa_furada_com_friso(), flat_shading=False)

In [ ]:
print("angular_tolerance mais fina — malha mais densa nas faces curvas")
show(caixa_furada_com_friso(), angular_tolerance=0.02)

## 3. `@interactive()` — sliders básicos

Ao decorar a função, os sliders e o modelo renderizado aparecem imediatamente. Arraste largura/altura e solte para disparar a reconstrução (padrão `continuous_update=False`).

In [ ]:
@interactive(width=(1, 10, 0.5, 5), height=(1, 8, 0.5, 3))
def modelo_caixa(width, height):
    return b3d.Box(width, height, 2)

A própria função decorada é devolvida sem alterações — continua chamável diretamente, sem o efeito colateral do widget:

In [ ]:
modelo_caixa(4, 4).volume

## 4. `@interactive()` com `show_kwargs`

Opções de exibição (cores, plano de base, etc.) são passadas em um dicionário `show_kwargs` separado, para nunca colidirem com o nome de um parâmetro do modelo.

In [ ]:
@interactive(
    width=(1, 10, 0.5, 5),
    height=(1, 8, 0.5, 3),
    show_kwargs=dict(colors=["steelblue"], z=0, plane_color="gainsboro"),
)
def caixa_com_plano(width, height):
    return b3d.Box(width, height, 2)

## 5. `@interactive()` com um controle `ipywidgets` explícito

Qualquer controle pode ser um widget já pronto, em vez de uma tupla `(min, max, step, default)` — útil para `Dropdown`, `Checkbox`, etc.

In [ ]:
@interactive(forma=widgets.Dropdown(options=["caixa", "cilindro"], value="caixa"))
def modelo_por_forma(forma):
    if forma == "caixa":
        return b3d.Box(4, 4, 4)
    return b3d.Cylinder(2, 4)

## 6. `@interactive()` com `continuous_update=True`

Reconstrução em tempo real a cada movimento do slider, em vez de apenas ao soltar — funciona bem aqui pois a caixa é barata de reconstruir/tesselar, mas pode travar em geometrias mais pesadas (ex.: a caixa furada com friso da seção 2).

In [ ]:
@interactive(width=(1, 10, 0.5, 5), continuous_update=True)
def caixa_ao_vivo(width):
    return b3d.Box(width, 3, 2)

## 7. `@interactive()` controlando uma diferença booleana

Combina as seções 2 e 3 — raio do furo controlado por slider em um corte booleano, verificando se a correção do sombreamento suave se mantém sob mudanças de parâmetro em tempo real.

In [ ]:
@interactive(raio_furo=(1, 9, 0.5, 5), raio_friso=(0.5, 4, 0.5, 2))
def modelo_com_corte(raio_furo, raio_friso):
    caixa = b3d.Box(20, 20, 20)
    furo = b3d.Cylinder(raio_furo, 30)
    cortada = caixa - furo
    arestas_verticais = cortada.edges().filter_by(b3d.Axis.Z)
    return b3d.fillet(arestas_verticais, radius=raio_friso)

## 8. `@interactive()` com retorno de dicionário — `show_kwargs` por quadro

Em vez de devolver só o(s) objeto(s), a função decorada pode devolver um `dict` com uma chave `"objects"` (o(s) objeto(s) a renderizar) e quaisquer outras chaves aceitas por `show()`/`_build_figure()` (`colors`, `opacity`, `z`, ...), que sobrescrevem `show_kwargs` **apenas naquele quadro** — útil, por exemplo, para sinalizar visualmente uma geometria inválida. Arraste o slider: abaixo de 5 a caixa fica azul (`steelblue`, o `show_kwargs` padrão); a partir de 5 fica vermelha (`indianred`, sobrescrita pelo dict retornado).

In [ ]:
@interactive(radius=(1, 9, 1, 5), show_kwargs=dict(colors=["steelblue"]))
def modelo_com_alerta(radius):
    box = b3d.Box(10, 10, 2)
    if radius >= 5:
        return {"objects": box, "colors": ["indianred"]}
    return box

---

## Resumo

Neste notebook você explorou:

- O parâmetro `flat_shading` do `show()` — sombreamento plano (facetado) vs. suave em faces curvas geradas por operações booleanas
- O parâmetro `angular_tolerance` — controla a densidade da malha nas faces curvas
- `@interactive()` — transforma qualquer função que devolve um sólido em um modelo com **sliders** (`(min, max, step, default)`), sem alterar a função em si
- `show_kwargs` — opções de exibição fixas, isoladas dos parâmetros do modelo
- Controles `ipywidgets` explícitos (`Dropdown`, etc.) como alternativa às tuplas de slider
- `continuous_update=True` — reconstrução em tempo real a cada movimento do slider
- Retorno de `dict` com `"objects"` — para sobrescrever `show_kwargs` **por quadro**, útil para sinalizar visualmente estados diferentes do modelo

---
*build123d para Arquitetos e Engenheiros — Versão Google Colab*